# <center>第四次作业</center>
## <center>金桐宇 BZ25219005</center>
### <center>SVHN 模型的 INT8 静态量化</center>

本报告在第二次作业 CNN-SVHN 分类模型的基础上完成训练后静态量化（PTQ）。实验比较 FP32 与 INT8 模型的测试精度、模型大小、CPU 单张图片推理延迟，并计算层输出量化误差。

## 实验说明

- 模型来源：使用 `../Homework 2/models.py` 中的 `SimpleCNN` 作为 FP32 基线模型。
- 权重来源：如果 `Homework 4/artifacts/hw2_simplecnn_fp32.pth` 不存在，本 notebook 会用作业二模型结构重新训练一个 FP32 模型，并将权重保存在第四次作业目录。
- 量化实现：手写 per-tensor 非对称线性量化/反量化函数；PTQ 转换使用 PyTorch eager mode 的 `QuantStub`、`DeQuantStub`、模块融合、校准和 `convert`。
- 环境限制：当前 PyTorch 只暴露 `onednn` INT8 后端，该后端要求 Conv/Linear 权重 zero point 为 0。因此实际 PTQ 配置为“激活 per-tensor affine 非对称，权重 per-tensor symmetric 对称”；手写量化函数仍按题目要求实现非对称线性量化。

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

from pathlib import Path
import pandas as pd
import torch

from homework4_quantization import (
    DEFAULT_CONFIG,
    HOMEWORK2_MODELS_PATH,
    ARTIFACTS_DIR,
    FIGURES_DIR,
    QuantizableSimpleCNN,
    load_homework2_models_module,
    linear_quantize,
    linear_dequantize,
    run_full_experiment,
)

print('Homework 2 model file:', HOMEWORK2_MODELS_PATH)
print('Artifacts dir:', ARTIFACTS_DIR)
print('Figures dir:', FIGURES_DIR)
print('Torch:', torch.__version__)
print('CUDA available for FP32 training:', torch.cuda.is_available())
print('Quantized engines:', torch.backends.quantized.supported_engines)

Homework 2 model file: D:\Microelectronics\neural-networks-and-their-applications\Homework 2\models.py
Artifacts dir: D:\Microelectronics\neural-networks-and-their-applications\Homework 4\artifacts
Figures dir: D:\Microelectronics\neural-networks-and-their-applications\Homework 4\Figures
Torch: 2.10.0+cu126
CUDA available for FP32 training: True
Quantized engines: ['onednn']


## 实验配置

默认配置使用完整 SVHN 测试集。训练 epoch、batch size、校准 batch 数等参数可通过环境变量覆盖，例如 `HW4_EPOCHS=12`。

In [2]:
config = dict(DEFAULT_CONFIG)
pd.DataFrame([config])

,batch_size,train_epochs,learning_rate,train_subset_size,test_subset_size,calibration_batches,error_batches,latency_warmup,latency_repeats,seed
0,256,8,0.001,None,None,16,4,30,200,42


## 手写线性量化与反量化

按照题目要求实现 per-tensor 非对称线性量化：先由张量最小值、最大值计算 scale 和 zero point，再执行 `round(x / scale + zero_point)` 并裁剪到整数范围。反量化使用 `(q - zero_point) * scale`。

In [3]:
x = torch.tensor([-1.20, -0.50, 0.00, 0.75, 1.80, 3.40], dtype=torch.float32)
q, scale, zero_point = linear_quantize(x, num_bits=8)
x_hat = linear_dequantize(q, scale, zero_point)
manual_quant_demo = pd.DataFrame({
    'x_fp32': x.numpy(),
    'q_uint8': q.numpy(),
    'x_dequant': x_hat.numpy(),
    'abs_error': (x - x_hat).abs().numpy(),
})
print(f'scale={scale:.8f}, zero_point={zero_point}')
manual_quant_demo

scale=0.01803922, zero_point=67


,x_fp32,q_uint8,x_dequant,abs_error
0,-1.20,0,-1.208627,0.008627
1,-0.50,39,-0.505098,0.005098
2,0.00,67,0.000000,0.000000
3,0.75,109,0.757647,0.007647
4,1.80,167,1.803922,0.003922
5,3.40,255,3.391372,0.008628


## 作业二模型复用与量化版包装

`Homework 2/models.py` 中的 `SimpleCNN` 使用函数式 ReLU，不便于 eager mode 的模块融合。因此这里在第四次作业中定义 `QuantizableSimpleCNN`：卷积层、全连接层和 dropout 与作业二 `SimpleCNN` 一致，并从作业二模型加载同名权重；额外加入显式 `ReLU` 模块以及 `QuantStub/DeQuantStub`，用于标记量化起止位置。

In [4]:
hw2_models = load_homework2_models_module()
hw2_model = hw2_models.SimpleCNN()
quantizable_model = QuantizableSimpleCNN(hw2_model)
print(hw2_model)
print('\nQuantizable wrapper:')
print(quantizable_model)

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4096, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

Quantizable wrapper:
QuantizableSimpleCNN(
  (quant): QuantStub()
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4096, out_features=512, bias=True)
  (relu3): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
  (dequant): DeQuantStub()
)


## FP32 训练、INT8 静态量化与评估

下面的单元会完成完整流程：

1. 下载/读取 SVHN 数据集。
2. 使用作业二 `SimpleCNN` 训练或加载 FP32 基线模型。
3. 构造量化版包装模型并加载 FP32 权重。
4. 在 CPU 上评估 FP32 精度、模型大小、单张图片平均延迟。
5. 融合 `Conv2d+ReLU`、`Linear+ReLU`，使用训练集样本校准并转换为 INT8 模型。
6. 在 CPU 上评估 INT8 精度、模型大小、单张图片平均延迟，并计算层输出 MSE。

In [5]:
results = run_full_experiment(config=config, force_train=False)
metrics = results['metrics']
errors = results['errors']
summary = results['summary']
metrics

,model,accuracy_percent,size_mb,latency_ms
0,FP32,89.751076,8.099467,0.484557
1,INT8 PTQ,89.678089,2.032422,0.693626


## 实验指标

In [6]:
metrics_display = metrics.copy()
metrics_display['accuracy_percent'] = metrics_display['accuracy_percent'].round(4)
metrics_display['size_mb'] = metrics_display['size_mb'].round(4)
metrics_display['latency_ms'] = metrics_display['latency_ms'].round(4)
print(f"量化后端: {summary['quant_engine']}")
print(f"精度损失: {summary['accuracy_loss_percent']:.4f}%")
print(f"压缩比: {summary['compression_ratio']:.4f}x")
print(f"CPU 推理加速比: {summary['speedup']:.4f}x")
metrics_display

量化后端: onednn
精度损失: 0.0730%
压缩比: 3.9851x
CPU 推理加速比: 0.6986x


,model,accuracy_percent,size_mb,latency_ms
0,FP32,89.7511,8.0995,0.4846
1,INT8 PTQ,89.6781,2.0324,0.6936


## 每层输出量化 MSE

In [7]:
errors_display = errors.copy()
errors_display['mse'] = errors_display['mse'].map(lambda v: f'{v:.8e}')
errors_display

,layer,mse
0,conv1,1.57802883e-05
1,conv2,2.59610060e-05
2,fc1,2.51949674e-04
3,fc2,1.24770659e-02
4,output,1.24770659e-02


## 可视化结果

![量化前后精度对比](Figures/accuracy_comparison.png)

![量化前后推理延迟对比](Figures/latency_comparison.png)

![每层输出量化 MSE](Figures/layer_mse.png)

## 结论

In [9]:
fp32 = metrics.iloc[0]
int8 = metrics.iloc[1]
print('本实验使用作业二 SimpleCNN 作为 FP32 基线，并通过 QuantStub/DeQuantStub、模块融合、校准和 convert 完成 INT8 PTQ。')
print(f"FP32 测试精度为 {fp32['accuracy_percent']:.2f}%，INT8 测试精度为 {int8['accuracy_percent']:.2f}%，精度损失 {summary['accuracy_loss_percent']:.2f}%。")
print(f"FP32 state_dict 大小为 {fp32['size_mb']:.3f} MB，INT8 state_dict 大小为 {int8['size_mb']:.3f} MB，压缩比 {summary['compression_ratio']:.2f}x。")
print(f"CPU 单张图片平均延迟由 {fp32['latency_ms']:.3f} ms 变为 {int8['latency_ms']:.3f} ms，加速比 {summary['speedup']:.2f}x。")
print('层输出 MSE 显示量化误差主要来自权重和激活离散化；若需要进一步降低精度损失，可增加校准样本、训练更高精度的作业二模型，或尝试量化感知训练 QAT。')

本实验使用作业二 SimpleCNN 作为 FP32 基线，并通过 QuantStub/DeQuantStub、模块融合、校准和 convert 完成 INT8 PTQ。
FP32 测试精度为 89.75%，INT8 测试精度为 89.68%，精度损失 0.07%。
FP32 state_dict 大小为 8.099 MB，INT8 state_dict 大小为 2.032 MB，压缩比 3.99x。
CPU 单张图片平均延迟由 0.485 ms 变为 0.694 ms，加速比 0.70x。
层输出 MSE 显示量化误差主要来自权重和激活离散化；若需要进一步降低精度损失，可增加校准样本、训练更高精度的作业二模型，或尝试量化感知训练 QAT。


## 复现方式

在仓库根目录使用指定 conda 环境执行：

```powershell
conda run -n neural_networks_and_their_applications jupyter nbconvert --to notebook --execute --inplace "Homework 4/assignment_4.ipynb"
```

如果当前目录已经是 `Homework 4`，则执行：

```powershell
conda run -n neural_networks_and_their_applications jupyter nbconvert --to notebook --execute --inplace assignment_4.ipynb
```

生成的模型、指标和图像保存在 `Homework 4/artifacts` 与 `Homework 4/Figures`。提交压缩包时可不包含 `Homework 4/dataset`。